In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = r"C:\Users\Papabizz\1. a Python\Ch 2\hospital_deterioration_hourly_panel.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head(3))


Shape: (417866, 28)
   patient_id  hour_from_admission  heart_rate  respiratory_rate  spo2_pct  \
0           1                    0       68.58             14.47     96.52   
1           1                    1       67.03             13.87     94.94   
2           1                    2       69.05             14.63     94.45   

   temperature_c  systolic_bp  diastolic_bp oxygen_device  oxygen_flow  ...  \
0          37.18       108.94         78.43          none          0.0  ...   
1          37.25       111.73         79.14          none          0.0  ...   
2          37.29       111.48         78.86          none          0.0  ...   

   age  gender  comorbidity_index  admission_type  baseline_risk_score  \
0   24       M                  2        Elective               0.2173   
1   24       M                  2        Elective               0.2173   
2   24       M                  2        Elective               0.2173   

   los_hours  deterioration_event  deterioration_with

In [3]:
print(df.shape)
df.info()

cat_cols = df.select_dtypes(include=["object","category","string"]).columns.tolist()
num_cols = df.select_dtypes(include=["number","bool","int64","float64"]).columns.tolist()
print("Categorical:", cat_cols)
print("Numeric count:", len(num_cols))

print("Total missing:", int(df.isna().sum().sum()))
print("Dup patient-hour:", df.duplicated(["patient_id","hour_from_admission"]).sum())

print("y rate (hourly):", df["deterioration_next_12h"].mean())
print("event rate (patient):", df.groupby("patient_id")["deterioration_event"].max().mean())


(417866, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417866 entries, 0 to 417865
Data columns (total 28 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   patient_id                               417866 non-null  int64  
 1   hour_from_admission                      417866 non-null  int64  
 2   heart_rate                               417866 non-null  float64
 3   respiratory_rate                         417866 non-null  float64
 4   spo2_pct                                 417866 non-null  float64
 5   temperature_c                            417866 non-null  float64
 6   systolic_bp                              417866 non-null  float64
 7   diastolic_bp                             417866 non-null  float64
 8   oxygen_device                            417866 non-null  object 
 9   oxygen_flow                              417866 non-null  float64
 10  mobility_score     

In [4]:
df = df.sort_values(["patient_id", "hour_from_admission"]).reset_index(drop=True)

# Confirm monotonic hours within each patient
non_mono = df.groupby("patient_id")["hour_from_admission"] \
             .apply(lambda s: not s.is_monotonic_increasing).sum()
print("Patients with non-monotonic time:", int(non_mono))



Patients with non-monotonic time: 0


In [5]:
mask_event = df["deterioration_hour"] >= 0
df = df[~(mask_event & (df["hour_from_admission"] >= df["deterioration_hour"]))].copy()

print("After dropping post-event rows:", df.shape)


After dropping post-event rows: (382548, 28)


In [6]:
patient_table = (
    df.groupby("patient_id", as_index=False)
      .agg(deterioration_event=("deterioration_event", "max"))
)

patient_ids = patient_table["patient_id"].to_numpy()
patient_strata = patient_table["deterioration_event"].to_numpy()

print("Total patients:", len(patient_ids))
print("Patient event rate (for stratify):", patient_strata.mean())


Total patients: 10000
Patient event rate (for stratify): 0.1938


In [7]:
from sklearn.model_selection import train_test_split

trainval_ids, test_ids = train_test_split(
    patient_ids,
    test_size=0.20,
    random_state=42,
    stratify=patient_strata
)

trainval_ids = set(trainval_ids)
test_ids = set(test_ids)

print("Trainval patients:", len(trainval_ids))
print("Test patients:", len(test_ids))


Trainval patients: 8000
Test patients: 2000


In [8]:
df_trainval = df[df["patient_id"].isin(trainval_ids)].copy()
df_test     = df[df["patient_id"].isin(test_ids)].copy()

overlap = set(df_trainval["patient_id"].unique()) & set(df_test["patient_id"].unique())
print("Patient overlap trainval vs test:", len(overlap))

print("Trainval rows:", df_trainval.shape)
print("Test rows:", df_test.shape)

print("Hourly y rate trainval:", df_trainval["deterioration_next_12h"].mean())
print("Hourly y rate test:", df_test["deterioration_next_12h"].mean())


Patient overlap trainval vs test: 0
Trainval rows: (306760, 28)
Test rows: (75788, 28)
Hourly y rate trainval: 0.05895814317381667
Hourly y rate test: 0.059415738639362434


In [9]:
mandatory_drop = [
    "deterioration_next_12h",
    "deterioration_hour",
    "deterioration_event",
    "deterioration_within_12h_from_admission",
    "los_hours",
]
optional_drop_clean = ["nurse_alert", "sepsis_risk_score", "baseline_risk_score"]

def make_Xy_clean(d):
    y = d["deterioration_next_12h"].astype(int)
    X = d.drop(columns=mandatory_drop + optional_drop_clean)
    groups = X["patient_id"].copy()      # keep for group CV later
    X = X.drop(columns=["patient_id"])   # not a model feature
    return X, y, groups

X_trainval, y_trainval, g_trainval = make_Xy_clean(df_trainval)
X_test, y_test, g_test             = make_Xy_clean(df_test)

print("X_trainval:", X_trainval.shape, "X_test:", X_test.shape)
print("y rate trainval:", y_trainval.mean(), "y rate test:", y_test.mean())


X_trainval: (306760, 19) X_test: (75788, 19)
y rate trainval: 0.05895814317381667 y rate test: 0.059415738639362434


In [10]:
pd.Series(sorted(list(trainval_ids))).to_csv("trainval_patient_ids_80.csv", index=False, header=["patient_id"])
pd.Series(sorted(list(test_ids))).to_csv("test_patient_ids_20.csv", index=False, header=["patient_id"])
print("Saved split patient IDs.")


Saved split patient IDs.


cross validation splitter object

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


In [12]:
cat_cols = X_trainval.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [c for c in X_trainval.columns if c not in cat_cols]


print("Categorical columns:", cat_cols)
print("Numeric columns:", len(num_cols))


Categorical columns: ['oxygen_device', 'gender', 'admission_type']
Numeric columns: 16


Logistic regression

ONE HOT ENCODING

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Preprocessing: apply different transforms to different column groups
preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),                          # scale numeric columns
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),    # one-hot categorical columns
    ]
)

# Baseline classifier
lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",   # helps with class imbalance
    solver="lbfgs"             # robust default solver
)

# Pipeline: preprocessing -> model
lr_clf = Pipeline(steps=[
    ("preprocess", preprocess_lr),
    ("model", lr)
])


Logistic regression summary

In [14]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

lr_auprc, lr_auroc = [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    # Fold split
    X_tr, y_tr = X_trainval.iloc[tr_idx], y_trainval.iloc[tr_idx]
    X_va, y_va = X_trainval.iloc[va_idx], y_trainval.iloc[va_idx]

    # Fit preprocessing+model on fold-train only
    lr_clf.fit(X_tr, y_tr)

    # Predict probabilities on fold-validation
    p_va = lr_clf.predict_proba(X_va)[:, 1]

    # Evaluate
    auprc = average_precision_score(y_va, p_va)
    auroc = roc_auc_score(y_va, p_va)

    lr_auprc.append(auprc)
    lr_auroc.append(auroc)

    print(f"Fold {fold}: AUPRC={auprc:.4f}, AUROC={auroc:.4f}")

print("\nLogistic Regression CV Summary")
print(f"AUPRC mean±std: {np.mean(lr_auprc):.4f} ± {np.std(lr_auprc):.4f}")
print(f"AUROC mean±std: {np.mean(lr_auroc):.4f} ± {np.std(lr_auroc):.4f}")


Fold 1: AUPRC=0.6873, AUROC=0.9404
Fold 2: AUPRC=0.6801, AUROC=0.9394
Fold 3: AUPRC=0.6347, AUROC=0.9299
Fold 4: AUPRC=0.6439, AUROC=0.9317
Fold 5: AUPRC=0.6064, AUROC=0.9260

Logistic Regression CV Summary
AUPRC mean±std: 0.6505 ± 0.0299
AUROC mean±std: 0.9335 ± 0.0056


In [15]:
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    tr_p = set(g_trainval.iloc[tr_idx].unique())
    va_p = set(g_trainval.iloc[va_idx].unique())
    print(f"Fold {fold}: patient overlap train∩val = {len(tr_p & va_p)}")


Fold 1: patient overlap train∩val = 0
Fold 2: patient overlap train∩val = 0
Fold 3: patient overlap train∩val = 0
Fold 4: patient overlap train∩val = 0
Fold 5: patient overlap train∩val = 0


In [16]:
results = []

def add_result(name, auprc_list, auroc_list):
    import numpy as np
    results.append({
        "model": name,
        "AUPRC_mean": float(np.mean(auprc_list)),
        "AUPRC_std":  float(np.std(auprc_list)),
        "AUROC_mean": float(np.mean(auroc_list)),
        "AUROC_std":  float(np.std(auroc_list)),
    })

# After LR:
add_result("LogReg", lr_auprc, lr_auroc)

import pandas as pd
pd.DataFrame(results).sort_values("AUPRC_mean", ascending=False)


,model,AUPRC_mean,AUPRC_std,AUROC_mean,AUROC_std
0,LogReg,0.650497,0.029891,0.933482,0.005575


Random forest

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# One-hot encode categoricals; keep numerics as-is
preprocess_rf = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

rf = RandomForestClassifier(
    n_estimators=200,          # strong baseline; 
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",   # helps with imbalance
    min_samples_leaf=2         # mild regularization to reduce overfitting
)

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess_rf),
    ("model", rf)
])


In [18]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

rf_auprc, rf_auroc = [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    X_tr, y_tr = X_trainval.iloc[tr_idx], y_trainval.iloc[tr_idx]
    X_va, y_va = X_trainval.iloc[va_idx], y_trainval.iloc[va_idx]

    rf_clf.fit(X_tr, y_tr)
    p_va = rf_clf.predict_proba(X_va)[:, 1]

    auprc = average_precision_score(y_va, p_va)
    auroc = roc_auc_score(y_va, p_va)

    rf_auprc.append(auprc)
    rf_auroc.append(auroc)

    print(f"Fold {fold}: AUPRC={auprc:.4f}, AUROC={auroc:.4f}")

print("\nRandom Forest CV Summary")
print(f"AUPRC mean±std: {np.mean(rf_auprc):.4f} ± {np.std(rf_auprc):.4f}")
print(f"AUROC mean±std: {np.mean(rf_auroc):.4f} ± {np.std(rf_auroc):.4f}")


Fold 1: AUPRC=0.6687, AUROC=0.9259
Fold 2: AUPRC=0.6494, AUROC=0.9237
Fold 3: AUPRC=0.6278, AUROC=0.9195
Fold 4: AUPRC=0.6424, AUROC=0.9146
Fold 5: AUPRC=0.5832, AUROC=0.9159

Random Forest CV Summary
AUPRC mean±std: 0.6343 ± 0.0287
AUROC mean±std: 0.9199 ± 0.0043


In [19]:
results.append({
    "model": "RandomForest",
    "AUPRC_mean": float(np.mean(rf_auprc)),
    "AUPRC_std":  float(np.std(rf_auprc)),
    "AUROC_mean": float(np.mean(rf_auroc)),
    "AUROC_std":  float(np.std(rf_auroc)),
})

import pandas as pd
pd.DataFrame(results).sort_values("AUPRC_mean", ascending=False)


,model,AUPRC_mean,AUPRC_std,AUROC_mean,AUROC_std
0,LogReg,0.650497,0.029891,0.933482,0.005575
1,RandomForest,0.634290,0.028726,0.919927,0.004347


Debug for boosting

In [20]:
import sys
print(sys.executable)
print(sys.version)


C:\Users\Papabizz\anaconda3\python.exe
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [21]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [22]:
import xgboost
print(xgboost.__version__)


3.1.3


In [23]:
%pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [24]:
from catboost import CatBoostClassifier
import catboost
print(catboost.__version__)


1.2.8


In [25]:
cat_cols = X_trainval.select_dtypes(include=["object", "category", "string"]).columns.tolist()
cat_features = [X_trainval.columns.get_loc(c) for c in cat_cols]

print("Categorical columns:", cat_cols)
print("Number of categorical columns:", len(cat_cols))


Categorical columns: ['oxygen_device', 'gender', 'admission_type']
Number of categorical columns: 3


In [26]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=800,          
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    thread_count=2,          # limit parallelism 
    task_type="CPU",
    verbose=0
)


In [27]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

cb_auprc, cb_auroc = [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    X_tr, y_tr = X_trainval.iloc[tr_idx], y_trainval.iloc[tr_idx]
    X_va, y_va = X_trainval.iloc[va_idx], y_trainval.iloc[va_idx]

    cb.fit(X_tr, y_tr, cat_features=cat_features)
    p_va = cb.predict_proba(X_va)[:, 1]

    auprc = average_precision_score(y_va, p_va)
    auroc = roc_auc_score(y_va, p_va)

    cb_auprc.append(auprc)
    cb_auroc.append(auroc)

    print(f"Fold {fold}: AUPRC={auprc:.4f}, AUROC={auroc:.4f}")

print("\nCatBoost CV Summary")
print(f"AUPRC mean±std: {np.mean(cb_auprc):.4f} ± {np.std(cb_auprc):.4f}")
print(f"AUROC mean±std: {np.mean(cb_auroc):.4f} ± {np.std(cb_auroc):.4f}")


Fold 1: AUPRC=0.6828, AUROC=0.9349
Fold 2: AUPRC=0.6856, AUROC=0.9393
Fold 3: AUPRC=0.6508, AUROC=0.9266
Fold 4: AUPRC=0.6692, AUROC=0.9300
Fold 5: AUPRC=0.6241, AUROC=0.9242

CatBoost CV Summary
AUPRC mean±std: 0.6625 ± 0.0228
AUROC mean±std: 0.9310 ± 0.0055


In [28]:
import pandas as pd

results = [
    {"model": "LogisticRegression", "AUPRC_mean": 0.6505, "AUPRC_std": 0.0299, "AUROC_mean": 0.9335, "AUROC_std": 0.0056},
    {"model": "RandomForest_200",   "AUPRC_mean": 0.6343, "AUPRC_std": 0.0287, "AUROC_mean": 0.9199, "AUROC_std": 0.0043},
    {"model": "CatBoost",           "AUPRC_mean": float(np.mean(cb_auprc)), "AUPRC_std": float(np.std(cb_auprc)),
                                     "AUROC_mean": float(np.mean(cb_auroc)), "AUROC_std": float(np.std(cb_auroc))},
]

pd.DataFrame(results).sort_values("AUPRC_mean", ascending=False)


,model,AUPRC_mean,AUPRC_std,AUROC_mean,AUROC_std
2,CatBoost,0.662486,0.022814,0.931007,0.005511
0,LogisticRegression,0.650500,0.029900,0.933500,0.005600
1,RandomForest_200,0.634300,0.028700,0.919900,0.004300


XGBoost

In [29]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# Identify categorical vs numeric columns 
cat_cols = X_trainval.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [c for c in X_trainval.columns if c not in cat_cols]

preprocess_xgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# Imbalance ratio for XGBoost
pos = int(y_trainval.sum())
neg = int(len(y_trainval) - pos)
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

xgb = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,                 # full parallelism
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight
)

xgb_clf = Pipeline(steps=[
    ("preprocess", preprocess_xgb),
    ("model", xgb)
])


scale_pos_weight: 15.96118544730731


In [30]:
from sklearn.metrics import average_precision_score, roc_auc_score

xgb_auprc, xgb_auroc = [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    X_tr, y_tr = X_trainval.iloc[tr_idx], y_trainval.iloc[tr_idx]
    X_va, y_va = X_trainval.iloc[va_idx], y_trainval.iloc[va_idx]

    xgb_clf.fit(X_tr, y_tr)
    p_va = xgb_clf.predict_proba(X_va)[:, 1]

    auprc = average_precision_score(y_va, p_va)
    auroc = roc_auc_score(y_va, p_va)

    xgb_auprc.append(auprc)
    xgb_auroc.append(auroc)

    print(f"Fold {fold}: AUPRC={auprc:.4f}, AUROC={auroc:.4f}")

import numpy as np
print("\nXGBoost CV Summary")
print(f"AUPRC mean±std: {np.mean(xgb_auprc):.4f} ± {np.std(xgb_auprc):.4f}")
print(f"AUROC mean±std: {np.mean(xgb_auroc):.4f} ± {np.std(xgb_auroc):.4f}")


Fold 1: AUPRC=0.6677, AUROC=0.9262
Fold 2: AUPRC=0.6562, AUROC=0.9317
Fold 3: AUPRC=0.6333, AUROC=0.9203
Fold 4: AUPRC=0.6412, AUROC=0.9181
Fold 5: AUPRC=0.5931, AUROC=0.9135

XGBoost CV Summary
AUPRC mean±std: 0.6383 ± 0.0255
AUROC mean±std: 0.9220 ± 0.0064


In [31]:
results.append({
    "model": "XGBoost",
    "AUPRC_mean": float(np.mean(xgb_auprc)),
    "AUPRC_std":  float(np.std(xgb_auprc)),
    "AUROC_mean": float(np.mean(xgb_auroc)),
    "AUROC_std":  float(np.std(xgb_auroc)),
})

pd.DataFrame(results).sort_values("AUPRC_mean", ascending=False)


,model,AUPRC_mean,AUPRC_std,AUROC_mean,AUROC_std
2,CatBoost,0.662486,0.022814,0.931007,0.005511
0,LogisticRegression,0.650500,0.029900,0.933500,0.005600
3,XGBoost,0.638262,0.025540,0.921978,0.006364
1,RandomForest_200,0.634300,0.028700,0.919900,0.004300


In [32]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

# Identify categorical vs numeric columns
cat_cols = X_trainval.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [c for c in X_trainval.columns if c not in cat_cols]

preprocess_hgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

hgb = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=600,
    random_state=42
)

hgb_clf = Pipeline(steps=[
    ("preprocess", preprocess_hgb),
    ("model", hgb)
])


In [33]:
def make_sample_weight(y):
    y = y.to_numpy()
    n = len(y)
    n_pos = int(y.sum())
    n_neg = n - n_pos
    w_pos = n / (2 * n_pos)
    w_neg = n / (2 * n_neg)
    return np.where(y == 1, w_pos, w_neg)


In [34]:
from sklearn.metrics import average_precision_score, roc_auc_score

hgb_auprc, hgb_auroc = [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval, groups=g_trainval), 1):
    X_tr, y_tr = X_trainval.iloc[tr_idx], y_trainval.iloc[tr_idx]
    X_va, y_va = X_trainval.iloc[va_idx], y_trainval.iloc[va_idx]

    w_tr = make_sample_weight(y_tr)

    # Fit on fold-train only (weights handle imbalance)
    hgb_clf.fit(X_tr, y_tr, model__sample_weight=w_tr)

    # Probability scores for metrics
    p_va = hgb_clf.predict_proba(X_va)[:, 1]

    auprc = average_precision_score(y_va, p_va)
    auroc = roc_auc_score(y_va, p_va)

    hgb_auprc.append(auprc)
    hgb_auroc.append(auroc)

    print(f"Fold {fold}: AUPRC={auprc:.4f}, AUROC={auroc:.4f}")

print("\nHistGradientBoosting CV Summary")
print(f"AUPRC mean±std: {np.mean(hgb_auprc):.4f} ± {np.std(hgb_auprc):.4f}")
print(f"AUROC mean±std: {np.mean(hgb_auroc):.4f} ± {np.std(hgb_auroc):.4f}")


Fold 1: AUPRC=0.6712, AUROC=0.9278
Fold 2: AUPRC=0.6561, AUROC=0.9305
Fold 3: AUPRC=0.6286, AUROC=0.9193
Fold 4: AUPRC=0.6384, AUROC=0.9173
Fold 5: AUPRC=0.5877, AUROC=0.9133

HistGradientBoosting CV Summary
AUPRC mean±std: 0.6364 ± 0.0284
AUROC mean±std: 0.9216 ± 0.0065


In [35]:
results.append({
    "model": "HistGradientBoosting",
    "AUPRC_mean": float(np.mean(hgb_auprc)),
    "AUPRC_std":  float(np.std(hgb_auprc)),
    "AUROC_mean": float(np.mean(hgb_auroc)),
    "AUROC_std":  float(np.std(hgb_auroc)),
})

import pandas as pd
pd.DataFrame(results).sort_values("AUPRC_mean", ascending=False)


,model,AUPRC_mean,AUPRC_std,AUROC_mean,AUROC_std
2,CatBoost,0.662486,0.022814,0.931007,0.005511
0,LogisticRegression,0.650500,0.029900,0.933500,0.005600
3,XGBoost,0.638262,0.025540,0.921978,0.006364
4,HistGradientBoosting,0.636385,0.028408,0.921631,0.006503
1,RandomForest_200,0.634300,0.028700,0.919900,0.004300


In [36]:
X_trainval.columns.tolist()

['hour_from_admission',
 'heart_rate',
 'respiratory_rate',
 'spo2_pct',
 'temperature_c',
 'systolic_bp',
 'diastolic_bp',
 'oxygen_device',
 'oxygen_flow',
 'mobility_score',
 'wbc_count',
 'lactate',
 'creatinine',
 'crp_level',
 'hemoglobin',
 'age',
 'gender',
 'comorbidity_index',
 'admission_type']